# 2.1 Building the feature matrix from lineage

**What this notebook is for.** Driving `trust_score_05.features` end to end on a
small slice, so you can see the intermediate frames rather than only the final
matrix. The production path is a Spark job; this is the same code with the lid
off.

**The shape of the pipeline.** Five stages, in this order, and the order matters:

1. **Reference population.** Given a reference timestamp, which entities existed?
   `build_reference_df_at_timestamp` answers it from the lineage table.
2. **Scope snapshots.** `build_scope_snapshots` produces one frame per scope
   (`msisdn`, `account`, `customer`) plus a `"base"` frame that is one row per
   lineage *interval*. Read the note below about `"base"` — it is the single
   easiest thing to get wrong in this package.
3. **Canonical events.** `build_changes_table` unions the account-change feed, the
   derived phone-number-change and MNO-port events, and the normalised EnStream
   API calls into one long table with a `event_family` column.
4. **Attach.** `attach_events_to_scope` left-joins events onto a scope's entities,
   bounded by the reference timestamp and the widest lookback window. Left, so a
   silent entity keeps its row.
5. **Assemble.** `assemble_scope_features` runs every calculator for every family
   and joins the results.

**The `"base"` trap.** `snapshot_scope["customer"]` is *aggregated*: one row per
customer, carrying `min(from_ts)` and `max(to_ts)`, and only the customer id
survives as an entity column. `snapshot_scope["base"]` is *unaggregated*: one row
per assignment interval, carrying every entity column. Any feature about an
entity's history or breadth must read `"base"`. Reading the scope entry instead
produced two shipped defects — finding 4.4 in
`trust_score_05/features/docs/feature_library_findings.md` — because both drafts
ran fine and emitted output of the right shape while being wrong. If you add a
calculator, read that finding first.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

In [ ]:
from trust_score_05.features import FeatureConfig, SCOPES

fcfg = FeatureConfig()

print("scopes           :", SCOPES)
print("lookback days    :", fcfg.lookback_days)
print("hour windows     :", fcfg.lookback_hour_windows)
print("ratio windows    :", fcfg.ratio_windows)
print("k event values   :", fcfg.k_event_values)
print("half life (days) :", fcfg.recency_half_life_days)
print("families         :", len(fcfg.generic_event_families))

In [ ]:
from trust_score_05.ml.jobs.base import build_spark_session

spark = build_spark_session("notebook")
spark.sparkContext.setLogLevel("WARN")
spark

## Inputs

Three frames come from the lineage layer. Point these at a *small* slice —
`trust_score_05/lineage/slice.py` exists for exactly this. The full lineage table
will not finish in a notebook.

- **lineage**: one row per `(msisdn, account, customer)` assignment interval, with
  `from_ts` / `to_ts`. This is the spine.
- **account changes**: the raw change feed.
- **EnStream calls**: the API-call log, hashed.

In [ ]:
# Set these three to real paths, or leave them as None and the next cell will
# build a tiny synthetic slice so the notebook runs end to end without cluster
# access.
LINEAGE_PATH = None
ACCOUNT_CHANGES_PATH = None
ENSTREAM_PATH = None

REFERENCE_TS = "2026-06-01 00:00:00"

In [ ]:
import datetime as dt

from pyspark.sql import types as T


def synthetic_inputs(spark):
    """A five-customer slice, small enough to print, wide enough to exercise
    every code path: a customer with two accounts, a customer who ported, a
    customer with an EnStream history, and a customer with no events at all."""
    lineage_schema = T.StructType(
        [
            T.StructField("msisdn", T.StringType()),
            T.StructField("acct_id", T.StringType()),
            T.StructField("customer_id", T.StringType()),
            T.StructField("mno", T.StringType()),
            T.StructField("from_ts", T.TimestampType()),
            T.StructField("to_ts", T.TimestampType()),
        ]
    )

    def ts(text):
        return dt.datetime.fromisoformat(text)

    far = ts("2999-12-31 00:00:00")
    lineage_rows = [
        ("P1", "A1", "C1", "BELL", ts("2021-06-01 00:00:00"), far),
        ("P2", "A2", "C2", "ROGERS", ts("2023-01-15 00:00:00"), far),
        ("P3", "A3", "C2", "ROGERS", ts("2024-03-01 00:00:00"), far),
        ("P4", "A4", "C3", "ROGERS", ts("2023-06-01 00:00:00"), ts("2026-02-01 00:00:00")),
        ("P4", "A5", "C3", "TELUS", ts("2026-02-01 00:00:00"), far),
        ("P5", "A6", "C4", "BELL", ts("2025-11-01 00:00:00"), far),
        ("P6", "A7", "C5", "BELL", ts("2021-08-01 00:00:00"), far),
    ]
    lineage = spark.createDataFrame(lineage_rows, lineage_schema)

    changes_schema = T.StructType(
        [
            T.StructField("msisdn", T.StringType()),
            T.StructField("event_ts", T.TimestampType()),
            T.StructField("event_type", T.StringType()),
        ]
    )
    changes_rows = [
        ("P1", ts("2026-05-20 10:00:00"), "DEVICE_CHANGE"),
        ("P1", ts("2026-05-25 10:00:00"), "SIM_CHANGE"),
        ("P1", ts("2026-05-29 10:00:00"), "STATUS_CHANGE-C"),
        ("P2", ts("2026-04-01 10:00:00"), "DEVICE_CHANGE"),
        ("P4", ts("2026-05-30 10:00:00"), "DEVICE_CHANGE"),
        ("P4", ts("2026-05-31 10:00:00"), "SIM_CHANGE"),
        ("P5", ts("2026-05-31 12:00:00"), "ACCOUNT_ACTIVATION"),
    ]
    changes = spark.createDataFrame(changes_rows, changes_schema)

    enstream_schema = T.StructType(
        [
            T.StructField("msisdn", T.StringType()),
            T.StructField("event_ts", T.TimestampType()),
            T.StructField("service_provider_id", T.StringType()),
            T.StructField("service_provider_name", T.StringType()),
            T.StructField("industry", T.StringType()),
            T.StructField("use_case", T.StringType()),
        ]
    )
    enstream_rows = [
        ("P1", ts("2026-05-10 09:00:00"), "SP1", "BankCo", "BANKING", "IDENTITY"),
        ("P1", ts("2026-05-28 09:00:00"), "SP1", "BankCo", "BANKING", "IDENTITY"),
        ("P1", ts("2026-05-30 09:00:00"), "SP2", "LendCo", "LENDING", "CREDIT"),
        ("P2", ts("2026-05-15 09:00:00"), "SP1", "BankCo", "BANKING", "IDENTITY"),
        ("P4", ts("2026-05-29 09:00:00"), "SP3", "TelCo", "TELECOM", "PORT_CHECK"),
    ]
    enstream = spark.createDataFrame(enstream_rows, enstream_schema)
    return lineage, changes, enstream


if LINEAGE_PATH is None:
    lineage_df, account_changes_df, enstream_df = synthetic_inputs(spark)
    print("using the synthetic slice")
else:
    lineage_df = spark.read.parquet(LINEAGE_PATH)
    account_changes_df = spark.read.parquet(ACCOUNT_CHANGES_PATH)
    enstream_df = spark.read.parquet(ENSTREAM_PATH)
    print("using the configured paths")

print("lineage rows :", lineage_df.count())
print("change rows  :", account_changes_df.count())
print("enstream rows:", enstream_df.count())
lineage_df.show(truncate=False)

## Stage 1 — the reference population

`build_reference_df_at_timestamp` asks: at this instant, which entities were
live? The interval test is closed at both ends and compares *dates*, not
timestamps, because a reference moment is trusted to the day and not to the
second. That is deliberate and documented in `snapshots.py`; the consequence is
that an entity whose assignment ended on the reference date is still in the
population, which is what you want for a same-day handover.

In [ ]:
import datetime as dt

from trust_score_05.features import build_reference_df

reference_ts = dt.datetime.fromisoformat(REFERENCE_TS)
references = build_reference_df(lineage_df, reference_ts=reference_ts, cfg=fcfg)

print("reference rows:", references.count())
references.show(truncate=False)

## Stage 2 — the scope snapshots

Note the row counts. There are more accounts than customers, and the `"base"`
frame has more rows than any scope, because it has not been aggregated. If your
`"base"` count equals your scope count, something has collapsed that should not
have.

In [ ]:
from trust_score_05.features import build_scope_snapshots

snapshot_scope = build_scope_snapshots(lineage_df, references, fcfg)

for key in ["base", *SCOPES]:
    frame = snapshot_scope[key]
    print(f"{key:10s} {frame.count():4d} rows   {frame.columns}")

print()
print("base -- one row per interval, every entity column present:")
snapshot_scope["base"].show(truncate=False)
print("customer -- one row per customer, aggregated:")
snapshot_scope["customer"].show(truncate=False)

## Stage 3 — the canonical event table

Four feeds become one table:

- `normalize_account_change_events` maps the raw feed onto the canonical family
  names in `CANONICAL_EVENT_FAMILY` and range-joins each event onto the lineage
  interval that was live when it happened. The upper bound of that join is
  *open*, so an event on a handover day is attributed to exactly one interval
  rather than both.
- `build_phone_number_change_events` derives a change event from the lineage
  itself, timestamped at the *arriving* row. The Gold implementation timestamped
  it at the departing row, which put the event before it happened — finding 1.1
  in `known_issues_in_previous_code.md`.
- `build_mno_port_events` derives ports, again from lineage, with no window bound.
  The Gold implementation bounded it, which meant a quiet port was never detected
  at all — finding 1.5 in the same document.
- `normalize_enstream_events` maps the API-call log in, carrying the partner
  metadata columns that the EnStream calculators grain on.

In [ ]:
from trust_score_05.features import (
    build_changes_table,
    build_mno_port_events,
    build_phone_number_change_events,
    normalize_account_change_events,
    normalize_enstream_events,
)

normalized_changes = normalize_account_change_events(account_changes_df, lineage_df, fcfg)
number_changes = build_phone_number_change_events(lineage_df, fcfg)
ports = build_mno_port_events(lineage_df, fcfg)
normalized_enstream = normalize_enstream_events(enstream_df, lineage_df, fcfg)

changes_df = build_changes_table(
    account_changes_df=normalized_changes,
    lineage_df=lineage_df,
    normalized_enstream_events_df=normalized_enstream,
    extra_event_frames=[number_changes, ports],
    cfg=fcfg,
)

print("canonical event rows:", changes_df.count())
changes_df.groupBy("event_family").count().orderBy("event_family").show(truncate=False)

## Stage 4 — attach

The join is a **left** join and the time condition is
`event_ts IS NULL OR (event_ts <= reference_ts AND event_ts >= lower_bound)`.
The `IS NULL` disjunct is what keeps an entity with no events in the frame at
all; without it the whole silent population vanishes and the feature matrix
quietly shrinks.

`cfg.prune_event_window` sets the lower bound to the widest configured lookback,
so events older than any window are dropped at the join rather than carried
through every calculator. That is a performance decision with no effect on
output.

In [ ]:
from trust_score_05.features import attach_events_to_scope

SCOPE = "customer"

scoped = attach_events_to_scope(snapshot_scope, changes_df, SCOPE, fcfg)
scoped.cache()

print("scoped rows :", scoped.count())
print("entities    :", scoped.select(*fcfg.entity_cols_for(SCOPE)).distinct().count())
scoped.select(*fcfg.entity_cols_for(SCOPE), "event_ts", "event_family").orderBy(
    *fcfg.entity_cols_for(SCOPE), "event_ts"
).show(30, truncate=False)

## Stage 5 — assemble

`assemble_scope_features` does stages 4 and 5 together — do not pass it the
already-scoped frame, pass it `snapshot_scope` and `changes_df`. It attaches
once, for all families, rather than once per family; that was finding 2.1, and
`tests/features/test_assemble.py` pins it by counting calls.

Two things it always does regardless of arguments: the stability features (tenure
and breadth, from `"base"`), and the EnStream partner features when an EnStream
frame is passed. State transitions are only computed if you ask for them.

In [ ]:
from trust_score_05.features import DEFAULT_STATE_TRANSITIONS, assemble_scope_features

features_df = assemble_scope_features(
    snapshot_scope=snapshot_scope,
    changes_df=changes_df,
    normalized_enstream_events_df=normalized_enstream,
    scope=SCOPE,
    state_transitions=DEFAULT_STATE_TRANSITIONS,
    event_families=None,  # None means every family in cfg.generic_event_families
    cfg=fcfg,
)
features_df.cache()

from trust_score_05.features.assemble import feature_column_names

names = feature_column_names(features_df, SCOPE, fcfg)
print("rows     :", features_df.count())
print("features :", len(names))

## Look at what came out

Group the names by family and calculator so the shape is legible. A count in the
hundreds is normal: eight calculators times a dozen families times several
windows multiplies quickly.

**Nulls are meaningful here.** A `null` count means the entity has never had an
event of that family; a `0` means it has, but not in the window. That asymmetry
is finding 4.5 and it is deliberate — see the findings document before
"fixing" it.

In [ ]:
import collections
import re

groups = collections.Counter()
for name in names:
    stem = re.sub(r"_\d+d(_over_\d+d)?$", "", name)
    stem = re.sub(r"_(cnt|any|frac|std|min|max|median|stddev|cv|slope)$", "", stem)
    groups[stem] += 1

for stem, count in sorted(groups.items()):
    print(f"{count:3d}  {stem}")

In [ ]:
pdf = features_df.limit(50).toPandas()
print("shape:", pdf.shape)
print()
print("null rate per column (top 20):")
print((pdf[names].isna().mean().sort_values(ascending=False).head(20) * 100).round(1).to_string())

## All three scopes at once

`assemble_all_scope_features` returns a **dict** keyed by scope, deliberately not
joined. Joining msisdn-grained features onto customer-grained rows is a decision
with a cardinality consequence, so the library makes you make it rather than
guessing. The ML side consumes one scope at a time.

In [ ]:
from trust_score_05.features import assemble_all_scope_features

all_scopes = assemble_all_scope_features(
    snapshot_scope=snapshot_scope,
    changes_df=changes_df,
    normalized_enstream_events_df=normalized_enstream,
    scopes=SCOPES,
    state_transitions=DEFAULT_STATE_TRANSITIONS,
    event_families=None,
    cfg=fcfg,
)

for scope, frame in all_scopes.items():
    cols = feature_column_names(frame, scope, fcfg)
    print(f"{scope:10s} {frame.count():4d} rows  {len(cols):4d} features  prefix={cols[0].split('_')[0]}")

## What next

- Adding a calculator? Read finding 4.4 first, add it to `calculators.py`, export
  it from `features/__init__.py`, and add a test to `tests/features/`. The
  existing tests are the template: each one names the defect it guards in its
  docstring.
- Writing this matrix out for real? That is a Spark job, not a notebook. The
  scale is the whole point and a notebook will not survive it.
- Next notebook: `2.2-feature-selection.ipynb`.